# 04 · Data drift y model drift con el segundo lote

Proyecto «Playlist perfecta para viajes en grupo» · NPD Entertainment · CEBANC · Ander Ameztoy

El lote 1 son las escuchas hasta el 30 de junio de 2025 y el lote 2 las posteriores (16 meses, ingeridas por el mismo `pipeline.sh`). Este notebook responde a tres preguntas:

1. **Data drift:** ¿han cambiado los hábitos de escucha entre los dos lotes (cuándo se escucha, cómo terminan las canciones, qué se escucha, cuánto se salta)?
2. **Model drift:** ¿el modelo del objetivo 1 entrenado con el lote 1 (la Ridge del notebook 03) se equivoca más a medida que pasa el tiempo, y en qué dirección?
3. **Estrategia:** ¿reentrenar arregla el problema? Se comparan tres políticas sobre el mismo periodo de prueba y se propone cómo vigilarlo y cuándo reentrenar.

Las métricas se registran en MLflow (experimento `drift`). Uso de IA (nivel VERDE): notebook escrito con ayuda de Claude y revisado por Ander.

## 0. Configuración

In [ ]:
from pathlib import Path
import os, json, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp, chi2_contingency
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
warnings.filterwarnings("ignore", category=UserWarning)

try:
    from google.colab import drive
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    drive.mount("/content/drive")
    RAIZ = Path("/content/drive/MyDrive/proyecto-Ebaluacion1")
else:
    RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

HISTORIALES = {
    "ander":  RAIZ / "data" / "entrada" / "historial" / "ander",
    "daniel": RAIZ / "data" / "entrada" / "historial" / "daniel",
}
CAMPOS = ["ts", "ms_played", "spotify_track_uri", "master_metadata_track_name", "master_metadata_album_artist_name",
          "master_metadata_album_album_name", "reason_end", "shuffle", "platform"]
ZONA = "Europe/Madrid"
CORTE = pd.Timestamp("2025-07-01", tz=ZONA)          # lote 1 < corte <= lote 2
CORTE_PRUEBA = pd.Timestamp("2026-01-01", tz=ZONA)   # experimento de reentrenamiento (sección 4)
MIN_ESCUCHAS, UMBRAL_SALTO_MS, SEMILLA = 5, 30_000, 42

COLOR = {"ander": "#56BE91", "daniel": "#FD3B1F"}
NEGRO, GRIS = "#020202", "#B9B6AE"
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "axes.titlelocation": "left"})
FIGURAS = RAIZ / "notebooks" / "figuras"; FIGURAS.mkdir(parents=True, exist_ok=True)
SALIDA = RAIZ / "data" / "salida"; SALIDA.mkdir(parents=True, exist_ok=True)
def guardar(fig, nombre):
    ruta = FIGURAS / f"{nombre}.png"; fig.savefig(ruta, dpi=200, bbox_inches="tight"); return ruta
print("Entorno:", "Google Colab" if EN_COLAB else "local", "| raíz:", RAIZ)

### MLflow

Igual que en el notebook 03: servidor del Docker si está levantado y, si no, la base SQLite `mlflow.db`. En Colab se parte de la copia que dejó el notebook 03 en `proyecto-Ebaluacion1/mlflow/`, para que los experimentos nuevos se sumen a los anteriores, y al final se vuelve a copiar allí.

In [ ]:
try:
    import mlflow
except ImportError:
    if EN_COLAB:
        os.system("pip install -q mlflow"); import mlflow
    else:
        mlflow = None; print("AVISO: mlflow no está instalado; los resultados se calculan igual pero no se registran.")

if mlflow is not None:
    uri = os.environ.get("MLFLOW_TRACKING_URI")
    if uri is None:
        try:
            import urllib.request
            urllib.request.urlopen("http://localhost:5000", timeout=2); uri = "http://localhost:5000"
        except Exception:
            carpeta_mlflow = Path("/content/mlflow") if EN_COLAB else RAIZ / "mlflow"
            carpeta_mlflow.mkdir(parents=True, exist_ok=True)
            previa = RAIZ / "mlflow" / "mlflow.db"
            if EN_COLAB and previa.exists() and not (carpeta_mlflow / "mlflow.db").exists():
                import shutil; shutil.copy2(previa, carpeta_mlflow / "mlflow.db")
                print("Partimos de los experimentos guardados en", previa)
            os.chdir(carpeta_mlflow)
            uri = f"sqlite:///{carpeta_mlflow / 'mlflow.db'}"
    mlflow.set_tracking_uri(uri); mlflow.set_experiment("drift")
    print("MLflow en:", uri)

def registrar(nombre, params, metricas, figuras=()):
    if mlflow is None: return
    with mlflow.start_run(run_name=nombre):
        mlflow.log_params(params)
        mlflow.log_metrics({k: float(v) for k, v in metricas.items() if pd.notna(v)})
        for f in figuras: mlflow.log_artifact(str(f))

## 1. Carga de los dos lotes

Se leen los historiales completos (los mismos JSON que ingiere `pipeline.sh`) con la limpieza mínima de silver que afecta a este análisis: solo música, sin reproducciones de 0 ms y sin duplicados de persona × instante × canción. Al final se comprueba que los recuentos del lote 1 cuadran con `gold.objetivo1`.

In [ ]:
def leer(carpeta, persona):
    ficheros = sorted(glob.glob(str(carpeta / "**" / "Streaming_History_Audio_*.json"), recursive=True))
    filas = []
    for f in ficheros:
        with open(f, encoding="utf-8") as fh:
            filas += [{k: r.get(k) for k in CAMPOS} for r in json.load(fh)]
    d = pd.DataFrame(filas, columns=CAMPOS); d["persona"] = persona
    return d

h = pd.concat([leer(c, p) for p, c in HISTORIALES.items()], ignore_index=True)
h["ms_played"] = pd.to_numeric(h["ms_played"])
h = h[h["spotify_track_uri"].notna() & (h["ms_played"] > 0)].copy()
h["track_id"] = h["spotify_track_uri"].str.replace("spotify:track:", "", regex=False)
h = h.sort_values("ms_played", ascending=False).drop_duplicates(["persona", "ts", "track_id"]).copy()
h["artista"] = h["master_metadata_album_artist_name"].fillna("?")
h["album"] = h["master_metadata_album_album_name"].fillna("?")
h["hora_local"] = pd.to_datetime(h["ts"], utc=True).dt.tz_convert(ZONA)
h["franja"] = h["hora_local"].dt.hour // 4                                   # 0-5, como en silver
h["tipo_dia"] = np.where(h["hora_local"].dt.dayofweek >= 5, "fin_semana", "laborable")
h["contexto"] = h["franja"].astype(str) + "·" + h["tipo_dia"]
h["lote"] = np.where(h["hora_local"] < CORTE, 1, 2)
h["mes"] = h["hora_local"].dt.tz_localize(None).dt.to_period("M")
h["no_salto"] = ~((h["reason_end"] == "fwdbtn") & (h["ms_played"] < UMBRAL_SALTO_MS))
h["shuffle"] = h["shuffle"].astype(bool)
h = h.sort_values("hora_local").reset_index(drop=True)

resumen = h.groupby(["persona", "lote"]).agg(
    reproducciones=("track_id", "size"), desde=("hora_local", "min"), hasta=("hora_local", "max"),
    meses=("mes", "nunique"), tasa_no_salto=("no_salto", "mean"))
resumen["desde"] = resumen["desde"].dt.date; resumen["hasta"] = resumen["hasta"].dt.date
resumen.round(3)

In [ ]:
# Comprobación con el gold (si está en data/dataset): escuchas del lote 1 por persona
ruta_gold = RAIZ / "data" / "dataset" / "gold_objetivo1.csv"
if ruta_gold.exists():
    g = pd.read_csv(ruta_gold, encoding="utf-8-sig")
    tot_gold = g.groupby(["persona", "lote"])["n_escuchas"].sum().unstack()
    tot_aqui = h.groupby(["persona", "lote"]).size().unstack()
    # el gold usa identificadores seudónimos: se emparejan por volumen de escuchas del lote 1
    aqui = tot_aqui.sort_values(1).reset_index()
    gold = tot_gold.sort_values(1).reset_index()
    emparejado = pd.DataFrame({"persona": aqui["persona"], "id en el gold": gold["persona"],
                               "lote 1 aquí": aqui[1], "lote 1 gold": gold[1], "lote 2 aquí": aqui[2], "lote 2 gold": gold[2]})
    emparejado["diferencia lote 1 (%)"] = (emparejado["lote 1 aquí"] / emparejado["lote 1 gold"] - 1).mul(100).round(1)
    display(emparejado.set_index("persona"))
else:
    print("No está gold_objetivo1.csv en data/dataset: se omite la comprobación.")

## 2. Data drift: ¿han cambiado los hábitos?

Para cada miembro se compara el lote 2 con dos referencias:

- **Lote 1 completo**, que es con lo que se entrena el modelo.
- **Últimos 12 meses del lote 1** (julio 2024 a junio 2025). Si el lote 2 solo se parece mal al lote 1 completo pero bien a su último año, el cambio es una tendencia lenta y no un salto reciente.

Medidas:

- **PSI** (*Population Stability Index*) de variables categóricas: contexto (franja × tipo de día, 12 categorías), cómo terminan las canciones (`reason_end`) y plataforma. Convenio habitual: < 0,10 estable, 0,10 a 0,25 cambio moderado, > 0,25 cambio importante.
- **KS** de la duración escuchada (`ms_played`).
- **Novedad:** parte de las escuchas del lote 2 que son de canciones o artistas que la persona no había escuchado nunca en el lote 1. Es lo que más afecta al modelo, porque para esas filas no hay historial de la canción.
- **Variable objetivo:** tasa de no salto, con un intervalo del 95 % por *bootstrap* por días (las escuchas de un mismo día no son independientes).

In [ ]:
def psi(ref, act, eps=1e-4):
    cats = sorted(set(ref.unique()) | set(act.unique()), key=str)
    p = ref.value_counts(normalize=True).reindex(cats, fill_value=0).to_numpy() + eps
    q = act.value_counts(normalize=True).reindex(cats, fill_value=0).to_numpy() + eps
    return float(np.sum((q - p) * np.log(q / p)))

def nivel(v):
    return "estable" if v < 0.10 else ("moderado" if v < 0.25 else "importante")

def ic_tasa(d, n=500, rng=np.random.default_rng(SEMILLA)):
    dias = d.groupby(d["hora_local"].dt.date)["no_salto"].agg(["sum", "size"]).to_numpy()
    b = [dias[i, 0].sum() / dias[i, 1].sum() for i in (rng.integers(0, len(dias), len(dias)) for _ in range(n))]
    return np.percentile(b, [2.5, 97.5])

h["plataforma"] = h["platform"].fillna("?").str.lower().str.extract(r"(android|ios|windows|osx|mac|web|linux|cast|partner|sonos)", expand=False).fillna("otra")
h["fin"] = h["reason_end"].where(h["reason_end"].isin(["trackdone", "fwdbtn", "endplay", "backbtn"]), "otros")

filas = []
for p, d in h.groupby("persona"):
    l1, l2 = d[d["lote"] == 1], d[d["lote"] == 2]
    reciente = l1[l1["hora_local"] >= CORTE - pd.DateOffset(years=1)]
    canciones_l1, artistas_l1 = set(l1["track_id"]), set(l1["artista"])
    for nombre_ref, ref in [("lote 1 completo", l1), ("últimos 12 meses del lote 1", reciente)]:
        ic1, ic2 = ic_tasa(ref), ic_tasa(l2)
        filas.append({"persona": p, "referencia": nombre_ref,
            "PSI contexto": psi(ref["contexto"], l2["contexto"]), "PSI reason_end": psi(ref["fin"], l2["fin"]),
            "PSI plataforma": psi(ref["plataforma"], l2["plataforma"]),
            "KS ms_played": ks_2samp(ref["ms_played"], l2["ms_played"]).statistic,
            "% shuffle ref": ref["shuffle"].mean() * 100, "% shuffle lote 2": l2["shuffle"].mean() * 100,
            "tasa no salto ref": ref["no_salto"].mean(), "IC ref": f"{ic1[0]:.3f}-{ic1[1]:.3f}",
            "tasa no salto lote 2": l2["no_salto"].mean(), "IC lote 2": f"{ic2[0]:.3f}-{ic2[1]:.3f}",
            "% escuchas de canciones nuevas": (~l2["track_id"].isin(canciones_l1)).mean() * 100,
            "% escuchas de artistas nuevos": (~l2["artista"].isin(artistas_l1)).mean() * 100})
deriva = pd.DataFrame(filas).set_index(["persona", "referencia"])
deriva.to_csv(SALIDA / "04_data_drift.csv")
deriva.round(3).T

Para separar si la novedad del lote 2 es «normal» (siempre se descubre música) o si ha crecido, se calcula lo mismo para el último año del lote 1 frente a todo lo anterior: es la tasa de novedad que el modelo ya «conocía».

In [ ]:
nov = []
for p, d in h.groupby("persona"):
    l1 = d[d["lote"] == 1]
    antes, ultimo = l1[l1["hora_local"] < CORTE - pd.DateOffset(years=1)], l1[l1["hora_local"] >= CORTE - pd.DateOffset(years=1)]
    l2 = d[d["lote"] == 2]
    nov.append({"persona": p,
                "último año del lote 1 frente a lo anterior (%)": (~ultimo["track_id"].isin(set(antes["track_id"]))).mean() * 100,
                "lote 2 frente al lote 1 (%)": (~l2["track_id"].isin(set(l1["track_id"]))).mean() * 100})
pd.DataFrame(nov).set_index("persona").round(1).rename_axis("escuchas de canciones nuevas")

In [ ]:
# Evolución mensual de la tasa de no salto y de la parte de escuchas de canciones nuevas (desde 2023)
desde = pd.Period("2023-01", "M")
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
for p, d in h[h["mes"] >= desde].groupby("persona"):
    m = d.groupby("mes").agg(tasa=("no_salto", "mean"), n=("no_salto", "size"))
    m = m[m["n"] >= 200]
    x = m.index.to_timestamp()
    axes[0].plot(x, m["tasa"], color=COLOR[p], lw=1.4, label=p.capitalize())
    axes[0].plot(x, m["tasa"].rolling(3, center=True).mean(), color=COLOR[p], lw=3, alpha=0.35)
    vistas, nuevas = set(h[(h["persona"] == p) & (h["mes"] < desde)]["track_id"]), []
    for mes, dm in d.groupby("mes"):
        nuevas.append((mes.to_timestamp(), (~dm["track_id"].isin(vistas)).mean() * 100))
        vistas |= set(dm["track_id"])
    nv = pd.DataFrame(nuevas, columns=["mes", "pct"]).set_index("mes")
    axes[1].plot(nv.index, nv["pct"], color=COLOR[p], lw=1.4, label=p.capitalize())
for ax in axes:
    ax.axvline(CORTE.tz_localize(None), color=NEGRO, ls="--", lw=1)
axes[0].text(CORTE.tz_localize(None), axes[0].get_ylim()[1], "  inicio del lote 2", va="top", fontsize=8.5)
axes[0].set_ylabel("tasa de no salto"); axes[0].set_title("Tasa de no salto por mes (línea gruesa: media de 3 meses)")
axes[1].set_ylabel("% escuchas de canciones\nnunca escuchadas antes"); axes[1].set_title("Descubrimiento de canciones nuevas por mes")
axes[0].legend(frameon=False)
fig.tight_layout(); fig_mensual = guardar(fig, "04_drift_mensual"); plt.show()

In [ ]:
# Distribución de contextos: lote 1 frente a lote 2 (cada barra es el % de escuchas de esa persona)
orden = [f"{f}·{t}" for t in ["laborable", "fin_semana"] for f in range(6)]
NOMBRES = ["0-4h", "4-8h", "8-12h", "12-16h", "16-20h", "20-24h"]
etiquetas = [f"{NOMBRES[int(o[0])]}\n{'lab.' if 'labor' in o else 'f. s.'}" for o in orden]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.3), sharey=True)
for ax, (p, d) in zip(axes, h.groupby("persona")):
    dist = d.groupby("lote")["contexto"].value_counts(normalize=True).mul(100).unstack(0).reindex(orden).fillna(0)
    x = np.arange(len(orden))
    ax.bar(x - 0.2, dist[1], 0.4, color=GRIS, label="lote 1")
    ax.bar(x + 0.2, dist[2], 0.4, color=COLOR[p], label="lote 2")
    ax.set_xticks(x); ax.set_xticklabels(etiquetas, fontsize=7)
    ax.set_title(f"{p.capitalize()} · PSI = {deriva.loc[(p, 'lote 1 completo'), 'PSI contexto']:.3f}", fontsize=10)
    ax.legend(frameon=False, fontsize=8)
axes[0].set_ylabel("% de escuchas")
fig.suptitle("Cuándo se escucha: contexto (franja × tipo de día)", x=0.01, ha="left", fontweight="bold")
fig.tight_layout(); fig_contexto = guardar(fig, "04_drift_contexto"); plt.show()

In [ ]:
for p in deriva.index.get_level_values(0).unique():
    r = deriva.loc[(p, "lote 1 completo")]
    registrar(f"data drift · {p}", {"persona": p, "referencia": "lote 1 completo", "lote_2": "2025-07-01 en adelante"},
              {"psi_contexto": r["PSI contexto"], "psi_reason_end": r["PSI reason_end"], "psi_plataforma": r["PSI plataforma"],
               "ks_ms_played": r["KS ms_played"], "tasa_no_salto_lote1": r["tasa no salto ref"],
               "tasa_no_salto_lote2": r["tasa no salto lote 2"], "pct_canciones_nuevas": r["% escuchas de canciones nuevas"]},
              [fig_mensual, fig_contexto])

## 3. Model drift: ¿el modelo se equivoca más con el tiempo?

Se reconstruye la tabla del objetivo 1 (persona × canción × contexto, con las mismas variables sin fuga del notebook 03) para cualquier periodo, a partir de las escuchas. Así se puede medir el error del modelo **por trimestre** dentro del lote 2, y no solo en el lote 2 entero.

- **Modelo vigilado:** la Ridge del notebook 03 (elegida por validación cruzada), entrenada con el lote 1 y congelada.
- **Error por trimestre:** MAE en las filas fiables (5 o más escuchas en ese trimestre) y **sesgo** (media de prevista menos real). Un sesgo positivo quiere decir que el modelo cree que se van a saltar menos canciones de las que se saltan de verdad.
- **Referencia:** el MAE de la validación cruzada en el lote 1, que es el error que se esperaba.

In [ ]:
def recuentos(d):
    """Escuchas y no saltos por persona × canción × contexto (los 'recuentos' de los que salen las variables)."""
    return (d.groupby(["persona", "track_id", "franja", "tipo_dia"])
             .agg(n_escuchas=("no_salto", "size"), n_no_salto=("no_salto", "sum"),
                  artista=("artista", "first"), album=("album", "first")).reset_index())

def tabla(etiquetas, fuente_d):
    """Filas con etiqueta de 'etiquetas' y variables de persona y grupo calculadas con 'fuente_d' (como gold.sql)."""
    t = recuentos(etiquetas)
    t["tasa_no_salto"] = t["n_no_salto"] / t["n_escuchas"]
    t["per_pct_shuffle"] = t["persona"].map(fuente_d.groupby("persona")["shuffle"].mean())
    pt = fuente_d.groupby(["persona", "track_id"])["no_salto"].agg(["size", "sum"])
    gt = pt.groupby("track_id").sum()
    own = pt.reindex(pd.MultiIndex.from_frame(t[["persona", "track_id"]])).fillna(0).to_numpy()
    tot = gt.reindex(t["track_id"]).fillna(0).to_numpy()
    n, ns = tot[:, 0] - own[:, 0], tot[:, 1] - own[:, 1]
    t["grp_n_escuchas"] = n
    t["grp_tasa_no_salto"] = np.where(n > 0, ns / np.maximum(n, 1), np.nan)
    return t

def tasa_suave(ns, n, previa, k=5):
    return (ns + k * previa) / (n + k)

def codificar(filas, src):
    f = filas[["persona", "track_id", "artista", "album", "franja", "tipo_dia"]].copy()
    per = src.groupby("persona")[["n_escuchas", "n_no_salto"]].sum()
    f["per_tasa"] = f["persona"].map(per["n_no_salto"] / per["n_escuchas"])
    ctx = src.groupby(["persona", "franja", "tipo_dia"])[["n_escuchas", "n_no_salto"]].sum()
    f = f.join((ctx["n_no_salto"] / ctx["n_escuchas"]).rename("ctx_tasa"), on=["persona", "franja", "tipo_dia"])
    propia = src.groupby(["persona", "track_id"])[["n_escuchas", "n_no_salto"]].sum()
    f = f.join(propia.rename(columns=lambda c: "own_" + c), on=["persona", "track_id"])
    for nombre, claves in [("art", ["persona", "artista"]), ("alb", ["persona", "artista", "album"])]:
        g = src.groupby(claves)[["n_escuchas", "n_no_salto"]].sum()
        f = f.join(g.rename(columns=lambda c: f"{nombre}_{c}_tot"), on=claves)
        n = (f[f"{nombre}_n_escuchas_tot"].fillna(0) - f["own_n_escuchas"].fillna(0)).clip(lower=0)
        ns = (f[f"{nombre}_n_no_salto_tot"].fillna(0) - f["own_n_no_salto"].fillna(0)).clip(lower=0)
        f[f"{nombre}_n"] = np.log1p(n)
        f[f"{nombre}_tasa"] = tasa_suave(ns, n, f["per_tasa"])
        f[f"{nombre}_conocido"] = (n > 0).astype(int)
    return f

def matriz(filas, cod):
    X = pd.DataFrame({
        "per_tasa": cod["per_tasa"], "ctx_tasa": cod["ctx_tasa"].fillna(cod["per_tasa"]),
        "art_tasa": cod["art_tasa"], "art_n": cod["art_n"], "art_conocido": cod["art_conocido"],
        "alb_tasa": cod["alb_tasa"], "alb_n": cod["alb_n"], "alb_conocido": cod["alb_conocido"],
        "per_pct_shuffle": filas["per_pct_shuffle"].to_numpy(),
        "grp_n": np.log1p(filas["grp_n_escuchas"].clip(lower=0)).to_numpy(),
        "grp_tasa": filas["grp_tasa_no_salto"].fillna(cod["per_tasa"]).to_numpy(),
        "grp_conocido": filas["grp_tasa_no_salto"].notna().astype(int).to_numpy(),
        "fin_semana": (filas["tipo_dia"] != "laborable").astype(int).to_numpy(),
    }, index=filas.index)
    for f in range(6):
        X[f"franja_{f}"] = (filas["franja"].astype(int) == f).astype(int).to_numpy()
    return X

def variables_oof(filas, src, k=5):
    partes = []
    for tr, te in GroupKFold(k).split(filas, groups=filas["track_id"]):
        sub = src[~src["track_id"].isin(set(filas.iloc[te]["track_id"]))]
        partes.append(matriz(filas.iloc[te], codificar(filas.iloc[te], sub)))
    return pd.concat(partes).loc[filas.index]

def ridge():
    return make_pipeline(StandardScaler(), Ridge(alpha=1.0))

def entrenar(periodo):
    """Entrena la Ridge con las filas fiables de 'periodo' (variables fuera de pliegue, pesos sqrt(n) como en el 03)."""
    t = tabla(periodo, periodo)
    t = t[t["n_escuchas"] >= MIN_ESCUCHAS].reset_index(drop=True)
    src = recuentos(periodo)
    X = variables_oof(t, src)
    m = ridge(); m.fit(X, t["tasa_no_salto"], ridge__sample_weight=np.sqrt(t["n_escuchas"]))
    return m, src, t, X

def evaluar(modelo, src, entreno_d, prueba_d, min_escuchas=MIN_ESCUCHAS):
    t = tabla(prueba_d, entreno_d)
    t = t[t["n_escuchas"] >= min_escuchas].reset_index(drop=True)
    X = matriz(t, codificar(t, src))
    t["prevista"] = np.clip(modelo.predict(X), 0, 1)
    t["base1"] = X["per_tasa"].to_numpy()
    t["cancion_nueva"] = X["art_conocido"].eq(0) & ~t.set_index(["persona", "track_id"]).index.isin(
        src.set_index(["persona", "track_id"]).index)
    return t

l1 = h[h["lote"] == 1]
modelo_v0, src_l1, t_l1, X_l1 = entrenar(l1)

# Error esperado: validación cruzada (GroupKFold por canción) en el lote 1, como en el notebook 03
oof = np.zeros(len(t_l1)); y = t_l1["tasa_no_salto"].to_numpy(); w = np.sqrt(t_l1["n_escuchas"].to_numpy())
for tr, te in GroupKFold(5).split(t_l1, groups=t_l1["track_id"]):
    src_tr = src_l1[src_l1["track_id"].isin(set(t_l1.iloc[tr]["track_id"]))]
    Xtr = variables_oof(t_l1.iloc[tr], src_tr)
    Xte = matriz(t_l1.iloc[te], codificar(t_l1.iloc[te], src_tr))
    m = ridge(); m.fit(Xtr, y[tr], ridge__sample_weight=w[tr])
    oof[te] = np.clip(m.predict(Xte), 0, 1)
MAE_ESPERADO = mean_absolute_error(y, oof)
print(f"Filas fiables del lote 1: {len(t_l1):,} | MAE esperado (validación cruzada): {MAE_ESPERADO:.3f}".replace(",", "."))

In [ ]:
# Error del modelo congelado por trimestre del lote 2
h["trimestre"] = h["hora_local"].dt.tz_localize(None).dt.to_period("Q")
l2 = h[h["lote"] == 2]
filas = []
for q, dq in l2.groupby("trimestre"):
    if dq["mes"].nunique() < 3: continue                     # trimestre incompleto (el último)
    ev = evaluar(modelo_v0, src_l1, l1, dq)
    for p, e in [("los dos", ev)] + list(ev.groupby("persona")):
        filas.append({"trimestre": str(q), "persona": p, "filas": len(e),
                      "MAE Ridge": mean_absolute_error(e["tasa_no_salto"], e["prevista"]),
                      "MAE modelo base 1": mean_absolute_error(e["tasa_no_salto"], e["base1"]),
                      "sesgo Ridge": (e["prevista"] - e["tasa_no_salto"]).mean(),
                      "tasa real": e["tasa_no_salto"].mean(), "tasa prevista": e["prevista"].mean()})
por_trim = pd.DataFrame(filas)
por_trim.to_csv(SALIDA / "04_model_drift_trimestres.csv", index=False)
por_trim.pivot(index="trimestre", columns="persona", values=["MAE Ridge", "sesgo Ridge", "filas"]).round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for p in ["ander", "daniel", "los dos"]:
    d = por_trim[por_trim["persona"] == p]
    estilo = dict(color=COLOR.get(p, NEGRO), lw=2.2 if p == "los dos" else 1.4, marker="o", ms=4, label=p.capitalize())
    axes[0].plot(d["trimestre"], d["MAE Ridge"], **estilo)
    axes[1].plot(d["trimestre"], d["sesgo Ridge"], **estilo)
axes[0].axhline(MAE_ESPERADO, color=GRIS, ls="--", lw=1.2); axes[0].text(0, MAE_ESPERADO, " MAE esperado (validación)", va="bottom", fontsize=8.5)
axes[1].axhline(0, color=GRIS, ls="--", lw=1.2)
axes[0].set_title("MAE de la Ridge congelada por trimestre"); axes[1].set_title("Sesgo (prevista - real)")
for ax in axes: ax.tick_params(axis="x", rotation=45, labelsize=8)
axes[0].legend(frameon=False, fontsize=8.5)
fig.tight_layout(); fig_trim = guardar(fig, "04_model_drift_trimestres"); plt.show()

tot = por_trim[por_trim["persona"] == "los dos"]
registrar("model drift · Ridge congelada (lote 1)", {"modelo": "Ridge alpha=1", "entrenado_con": "lote 1", "evaluacion": "trimestres del lote 2, filas con 5+ escuchas"},
          {"mae_esperado_cv": MAE_ESPERADO, "mae_lote2_medio": tot["MAE Ridge"].mean(), "mae_lote2_max": tot["MAE Ridge"].max(),
           "sesgo_lote2_medio": tot["sesgo Ridge"].mean(),
           **{f"mae_{t.replace('Q', 't')}": v for t, v in zip(tot["trimestre"], tot["MAE Ridge"])}}, [fig_trim])

### Calibración mes a mes: la señal para vigilar en producción

El MAE por trimestre solo usa canciones escuchadas 5 o más veces en ese trimestre, que son las que más gustan. Para vigilar el modelo en producción es más útil comparar, **cada mes y con todas las escuchas**, la tasa de no salto media que prevé el modelo con la real. No necesita esperar a tener filas fiables y separa bien los dos tipos de cambio que hay en los datos.

In [ ]:
filas = []
for mes, dm in l2.groupby("mes"):
    ev = evaluar(modelo_v0, src_l1, l1, dm, min_escuchas=1)
    for p, e in ev.groupby("persona"):
        if e["n_escuchas"].sum() < 300: continue                 # meses con muy pocas escuchas
        w = e["n_escuchas"]
        filas.append({"mes": str(mes), "persona": p, "escuchas": int(w.sum()),
                      "prevista": (e["prevista"] * w).sum() / w.sum(), "real": (e["tasa_no_salto"] * w).sum() / w.sum(),
                      "% shuffle": dm.loc[dm["persona"] == p, "shuffle"].mean() * 100})
calib = pd.DataFrame(filas)
calib["desvío"] = calib["prevista"] - calib["real"]
calib.to_csv(SALIDA / "04_calibracion_mensual.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharex=True)
for ax, (p, d) in zip(axes, calib.groupby("persona")):
    x = pd.PeriodIndex(d["mes"], freq="M").to_timestamp()
    ax.plot(x, d["real"], color=COLOR[p], lw=2, marker="o", ms=3, label="real")
    ax.plot(x, d["prevista"], color=NEGRO, lw=1.2, ls="--", label="prevista (Ridge del lote 1)")
    ax.fill_between(x, d["prevista"] - 0.05, d["prevista"] + 0.05, color=GRIS, alpha=0.3, label="banda de ±0,05")
    ax.set_title(p.capitalize(), fontsize=10); ax.tick_params(axis="x", rotation=45, labelsize=8)
axes[0].set_ylabel("tasa de no salto del mes"); axes[0].legend(frameon=False, fontsize=8)
fig.suptitle("Calibración mensual en el lote 2: tasa prevista frente a real", x=0.01, ha="left", fontweight="bold")
fig.tight_layout(); fig_calib = guardar(fig, "04_calibracion_mensual"); plt.show()
registrar("model drift · calibración mensual", {"modelo": "Ridge del lote 1", "medida": "tasa prevista - real, todas las escuchas del mes"},
          {f"desvio_{r.persona}_{r.mes}": r.desvío for r in calib.itertuples()} |
          {f"meses_fuera_banda_{p}": int((d["desvío"].abs() > 0.05).sum()) for p, d in calib.groupby("persona")}, [fig_calib])
calib.pivot(index="mes", columns="persona", values=["real", "prevista", "desvío", "% shuffle"]).round(3)

## 4. ¿Reentrenar lo arregla? Tres políticas sobre el mismo periodo de prueba

Periodo de prueba: **desde el 1 de enero de 2026** hasta el final del historial. Se comparan:

| Política | Entrena con | Qué representa |
| --- | --- | --- |
| Congelado | lote 1 (hasta junio 2025) | no hacer nada |
| Reentrenado acumulado | todo hasta diciembre 2025 | reentrenar cada 6 meses con todo el historial |
| Ventana de 18 meses | julio 2024 a diciembre 2025 | reentrenar olvidando lo antiguo |

Las variables de cada política (tasa de la persona, del artista, del grupo...) salen de su propio periodo de entrenamiento: reentrenar también actualiza el «perfil» de cada miembro, no solo los coeficientes.

In [ ]:
prueba = h[h["hora_local"] >= CORTE_PRUEBA]
POLITICAS = {
    "congelado (lote 1)": h[h["hora_local"] < CORTE],
    "reentrenado acumulado": h[h["hora_local"] < CORTE_PRUEBA],
    "ventana de 18 meses": h[(h["hora_local"] >= CORTE_PRUEBA - pd.DateOffset(months=18)) & (h["hora_local"] < CORTE_PRUEBA)],
}
filas, evals = [], {}
for nombre, periodo in POLITICAS.items():
    m, src, t, _ = entrenar(periodo)
    ev = evaluar(m, src, periodo, prueba)
    evals[nombre] = ev
    fila = {"política": nombre, "filas de entrenamiento": len(t), "filas de prueba": len(ev),
            "MAE": mean_absolute_error(ev["tasa_no_salto"], ev["prevista"]),
            "MAE modelo base 1": mean_absolute_error(ev["tasa_no_salto"], ev["base1"]),
            "sesgo": (ev["prevista"] - ev["tasa_no_salto"]).mean()}
    for p, e in ev.groupby("persona"):
        fila[f"MAE {p}"] = mean_absolute_error(e["tasa_no_salto"], e["prevista"])
        fila[f"sesgo {p}"] = (e["prevista"] - e["tasa_no_salto"]).mean()
    filas.append(fila)
    registrar(f"reentrenamiento · {nombre}", {"politica": nombre, "entrena_desde": str(periodo["hora_local"].min().date()),
              "entrena_hasta": str(periodo["hora_local"].max().date()), "prueba_desde": str(CORTE_PRUEBA.date()), "modelo": "Ridge alpha=1"},
              {k.lower().replace(" ", "_"): v for k, v in fila.items() if k not in ("política",)})
politicas = pd.DataFrame(filas).set_index("política")
politicas.to_csv(SALIDA / "04_politicas_reentrenamiento.csv")
politicas.round(3)

In [ ]:
# Las filas de prueba no son las mismas en las tres políticas (cambia qué canciones son "nuevas"), pero la etiqueta sí:
# comparación justa sobre las filas comunes
claves = ["persona", "track_id", "franja", "tipo_dia"]
comun = evals["congelado (lote 1)"][claves + ["tasa_no_salto"]]
for nombre, ev in evals.items():
    comun = comun.merge(ev[claves + ["prevista"]].rename(columns={"prevista": nombre}), on=claves)
justa = pd.DataFrame({n: [mean_absolute_error(comun["tasa_no_salto"], comun[n]), (comun[n] - comun["tasa_no_salto"]).mean()]
                      for n in POLITICAS}, index=["MAE (filas comunes)", "sesgo (filas comunes)"]).T
print(f"Filas comunes: {len(comun):,}".replace(",", "."))

fig, ax = plt.subplots(figsize=(7, 2.8))
colores = [GRIS, "#FD3B1F", "#56BE91"]
ax.barh(justa.index[::-1], justa["MAE (filas comunes)"][::-1], color=colores[::-1])
for i, v in enumerate(justa["MAE (filas comunes)"][::-1]):
    ax.text(v + 0.002, i, f"{v:.3f}", va="center", fontsize=9)
ax.axvline(MAE_ESPERADO, color=NEGRO, ls="--", lw=1); ax.text(MAE_ESPERADO, -0.75, " MAE esperado (validación)", fontsize=8.5)
ax.set_xlim(0, justa["MAE (filas comunes)"].max() * 1.25)
ax.set_title("MAE en 2026 según la política de reentrenamiento (menos es mejor)")
fig.tight_layout(); fig_pol = guardar(fig, "04_politicas_reentrenamiento"); plt.show()
if mlflow is not None:
    with mlflow.start_run(run_name="reentrenamiento · comparación justa"):
        mlflow.log_metrics({f"mae_{n.split()[0]}": v for n, v in justa["MAE (filas comunes)"].items()}); mlflow.log_artifact(str(fig_pol))
justa.round(3)

## 5. Conclusiones y estrategia de monitorización

Las conclusiones de esta sección se escriben a partir de las tablas anteriores; la celda siguiente las resume en números para el informe.

In [ ]:
r = {p: deriva.loc[(p, "lote 1 completo")] for p in deriva.index.get_level_values(0).unique()}
for p, v in r.items():
    print(f"{p.capitalize():7} tasa de no salto {v['tasa no salto ref']:.3f} -> {v['tasa no salto lote 2']:.3f} | "
          f"PSI contexto {v['PSI contexto']:.3f} ({nivel(v['PSI contexto'])}), reason_end {v['PSI reason_end']:.3f} ({nivel(v['PSI reason_end'])}), "
          f"plataforma {v['PSI plataforma']:.3f} ({nivel(v['PSI plataforma'])}) | canciones nuevas {v['% escuchas de canciones nuevas']:.1f} %")
print(f"\nMAE esperado {MAE_ESPERADO:.3f}; MAE por trimestre del lote 2 (los dos): "
      + ", ".join(f"{t} {m:.3f}" for t, m in zip(tot["trimestre"], tot["MAE Ridge"])))
print("MAE en 2026 (filas comunes): " + ", ".join(f"{n} {v:.3f}" for n, v in justa["MAE (filas comunes)"].items()))
json.dump({"data_drift": deriva.reset_index().to_dict("records"), "trimestres": por_trim.to_dict("records"),
           "politicas": politicas.reset_index().to_dict("records"), "politicas_filas_comunes": justa.reset_index().to_dict("records"),
           "mae_esperado": MAE_ESPERADO}, open(SALIDA / "04_resumen.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1, default=str)

### Qué dicen los datos

**1. Data drift: dos tipos de cambio distintos.**

- **Ander, cambio gradual en lo que se salta.** Su tasa de no salto baja de 0,552 en el lote 1 a 0,435 en el lote 2, y los intervalos del 95 % no se tocan. Pero frente a su último año del lote 1 (0,476) la caída es mucho menor: es sobre todo una **tendencia lenta** que ya venía de antes, no un salto el 1 de julio. *Cuándo* escucha casi no cambia (PSI de contexto 0,06) y descubre canciones al mismo ritmo que antes (34 % de escuchas de canciones nuevas, igual que en su último año). El PSI de plataforma frente al lote 1 completo es alto (0,72) porque cambió de dispositivo hace años; frente a su último año es 0,01, así que no afecta.
- **Daniel, un episodio corto y fuerte.** Hasta julio de 2026 está estable (tasa ≈ 0,94-0,96). En **agosto y septiembre de 2026** su tasa cae a 0,87 y 0,72, el `fwdbtn` sube al 30 % y el modo aleatorio baja del 75-80 % al 19 %: cambió su forma de escuchar (elige él las canciones y las salta más). En octubre y noviembre vuelve a su nivel. Su contexto sí se mueve algo (PSI 0,12, moderado): escucha más de madrugada y por la noche en días laborables.

**2. Model drift: el modelo está descalibrado para Ander desde el primer mes.**

- El error esperado era MAE 0,166 (validación en el lote 1); en el lote 2 está entre 0,21 y 0,26 todos los trimestres. Ya en el primer trimestre es un 30 % peor, así que el problema no aparece con el tiempo: es la diferencia entre el lote 1 completo y el presente.
- La calibración mensual lo explica: para **Ander** el modelo prevé una tasa de ≈ 0,55 todos los meses y la real está entre 0,35 y 0,52; queda **fuera de la banda de ±0,05 en 15 de 16 meses**. Para **Daniel** está dentro de la banda todos los meses salvo septiembre de 2026 (+0,18), el centro del episodio anterior; agosto se queda justo en el límite (+0,04).
- En el MAE por trimestre el sesgo de Ander sale cerca de 0 y no de +0,10 porque esa medida solo usa las canciones escuchadas 5 o más veces en el trimestre (sus favoritas, que salta poco). Por eso, para vigilar, la medida buena es la calibración mensual con todas las escuchas.
- Dato incómodo: en el lote 2 la Ridge no gana a la media de la persona (modelo base 1). Lo que aprende en el lote 1 (artista, contexto, grupo) se desgasta, y el mayor error viene del nivel general de la persona, que es lo primero que hay que actualizar.

**3. Reentrenar sí ayuda, y olvidar lo antiguo ayuda más.** En 2026 (mismas 4.759 filas para las tres políticas):

| Política | MAE | Sesgo |
| --- | --- | --- |
| Congelado (lote 1) | 0,225 | +0,080 |
| Reentrenado acumulado | 0,218 | +0,064 |
| Ventana de 18 meses | **0,207** | **+0,027** |

La ventana de 18 meses reduce el sesgo de Ander de +0,09 a +0,03, porque su tasa de referencia pasa a ser la reciente. El MAE sigue por encima del esperado (0,166), así que reentrenar corrige el nivel pero no basta: la mejora del modelo del objetivo 1 sigue pendiente (notebook 05).

### Estrategia de monitorización y reentrenamiento

| Qué se vigila | Cada cuánto | Umbral de alerta | Acción |
| --- | --- | --- | --- |
| Desvío de calibración por persona (tasa prevista - real del mes, todas las escuchas) | mensual | fuera de ±0,05 **dos meses seguidos** | reentrenar con la ventana de 18 meses |
| PSI de contexto y de `reason_end` por persona frente a los últimos 12 meses | mensual | > 0,25 | revisar los datos de esa persona antes de reentrenar (¿cambio de hábito o error de ingesta?) |
| % de modo aleatorio y de `fwdbtn` del mes | mensual | cambio de más de 20 puntos | marcar el mes como episodio: no reentrenar solo con él |
| MAE en filas fiables del trimestre | trimestral | > MAE esperado + 0,05 | reentrenar y volver a comparar modelos (notebook 05) |
| Reentrenamiento programado | cada 6 meses | — | ventana de 18 meses aunque no salte ninguna alerta |

- **Por qué dos meses seguidos:** el episodio de Daniel solo saca la calibración de la banda un mes y luego vuelve; con la regla de un mes se habría reentrenado sobre un comportamiento pasajero. Con la de dos meses, a Ander se le habría reentrenado ya en agosto de 2025 (julio +0,14 y agosto +0,09).
- **Ventana y no acumulado:** los gustos cambian despacio y el pasado lejano pesa demasiado en el acumulado (seis años de Ander frente a 16 meses recientes).
- **Cómo se haría en el proyecto:** cada lote nuevo entra por `pipeline.sh` (HDFS, silver y gold); este notebook calcula las métricas y las registra en MLflow (experimento `drift`). Si salta una alerta, se reentrena, se registra el modelo nuevo como versión nueva en el registro de modelos y solo pasa a producción si mejora al anterior en el último trimestre.

## 6. Guardar los experimentos de MLflow en Drive (solo Colab)

In [ ]:
if EN_COLAB and mlflow is not None and mlflow.get_tracking_uri().startswith("sqlite"):
    import shutil
    destino = RAIZ / "mlflow"; destino.mkdir(parents=True, exist_ok=True)
    shutil.copy2("/content/mlflow/mlflow.db", destino / "mlflow.db")
    if Path("/content/mlflow/mlartifacts").exists():
        shutil.copytree("/content/mlflow/mlartifacts", destino / "mlartifacts", dirs_exist_ok=True)
    print("Experimentos copiados a", destino)